In [ ]:
import json
import random

import numpy as np
from tqdm.auto import tqdm


/Users/annakobiakova/diploma/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TRAIN_PATH = "./data/train.txt"
DEV_PATH = "./data/dev.txt"
OUTPUT_PREPARED = "prepared_dev_6.jsonl"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABELS = [
    "ABBREVIATION", "ALTERNATIVE_NAME", "SUBCLASS_OF", "PART_OF",
    "TREATED_USING", "ORIGINS_FROM", "TO_DETECT_OR_STUDY", "AFFECTS",
    "HAS_CAUSE", "APPLIED_TO", "USED_IN", "ASSOCIATED_WITH",
    "PHYSIOLOGY_OF", "FINDING_OF",
    "no_relation"
]

K_SHOTS = 6


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_data = load_jsonl(TRAIN_PATH)
dev_data = load_jsonl(DEV_PATH)

print(f"Train loaded: {len(train_data)}")
print(f"Dev loaded:   {len(dev_data)}")

Train loaded: 12739
Dev loaded:   2967


In [ ]:
def get_head_name(item):
    return item["h"]["name"]

def get_tail_name(item):
    return item["t"]["name"]

def get_head_pos(item):
    return item["h"]["pos"]

def get_tail_pos(item):
    return item["t"]["pos"]

def get_head_type(item):
    return item["head_type"]

def get_tail_type(item):
    return item["tail_type"]

def get_relation(item):
    return item.get("relation", None)

def get_doc_id(item):
    return item.get("doc_id", "unknown")

def get_head_span(item):
    return item.get("head_span", "")

def get_tail_span(item):
    return item.get("tail_span", "")

def insert_markers_by_spans(text, h_pos, t_pos, h_type, t_type):
    hs, he = h_pos
    ts, te = t_pos

    spans = [
        (hs, he, f"[HEAD_{h_type}]", f"[/HEAD_{h_type}]"),
        (ts, te, f"[TAIL_{t_type}]", f"[/TAIL_{t_type}]"),
    ]

    spans = sorted(spans, key=lambda x: x[0], reverse=True)

    out = text
    for start, end, open_tag, close_tag in spans:
        out = out[:start] + f"{open_tag} " + out[start:end] + f" {close_tag}" + out[end:]
    return out

def extract_window_around_entities(text, h_pos, t_pos, window=200):
    hs, he = h_pos
    ts, te = t_pos

    left = max(0, min(hs, ts) - window)
    right = min(len(text), max(he, te) + window)

    cropped = text[left:right]
    new_h_pos = [hs - left, he - left]
    new_t_pos = [ts - left, te - left]

    return cropped, new_h_pos, new_t_pos

def build_marked_text(item, window=200):
    text = item["text"]
    h_pos = get_head_pos(item)
    t_pos = get_tail_pos(item)

    cropped_text, new_h_pos, new_t_pos = extract_window_around_entities(
        text, h_pos, t_pos, window=window
    )

    return insert_markers_by_spans(
        text=cropped_text,
        h_pos=new_h_pos,
        t_pos=new_t_pos,
        h_type=get_head_type(item),
        t_type=get_tail_type(item),
    )


def format_example(item, include_label=True):
    marked_text = build_marked_text(item)

    h_name = get_head_name(item)
    t_name = get_tail_name(item)
    h_type = get_head_type(item)
    t_type = get_tail_type(item)

    s = (
        f"Text: {marked_text}\n"
        f"Entity 1 (Head): {h_name} (Type: {h_type})\n"
        f"Entity 2 (Tail): {t_name} (Type: {t_type})\n"
    )
    if include_label:
        s += f"Relation: {get_relation(item)}\n"
    else:
        s += "Relation:"
    return s

In [ ]:
train_by_pair_type = {}
for item in train_data:
    key = (get_head_type(item), get_tail_type(item))
    train_by_pair_type.setdefault(key, []).append(item)

def lexical_similarity(a, b):
    wa = set(str(a).lower().split())
    wb = set(str(b).lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def same_instance(a, b):
    return (
        get_doc_id(a) == get_doc_id(b)
        and get_head_span(a) == get_head_span(b)
        and get_tail_span(a) == get_tail_span(b)
    )

def retrieve_few_shot_examples(query_item, k=K_SHOTS):
    h_type = get_head_type(query_item)
    t_type = get_tail_type(query_item)
    pair_key = (h_type, t_type)

    candidates = train_by_pair_type.get(pair_key, [])
    if len(candidates) < k:
        candidates = train_data

    scored = []
    q_text = query_item["text"]

    for cand in candidates:
        if same_instance(cand, query_item):
            continue
        sim = lexical_similarity(q_text, cand["text"])
        scored.append((sim, cand))

    scored.sort(key=lambda x: x[0], reverse=True)
    top = [x[1] for x in scored[:50]]

    selected = []
    used_rel = set()

    for ex in top:
        rel = get_relation(ex)
        if rel not in used_rel:
            selected.append(ex)
            used_rel.add(rel)
        if len(selected) >= k:
            break

    if len(selected) < k:
        for ex in top:
            if ex not in selected:
                selected.append(ex)
            if len(selected) >= k:
                break

    return selected[:k]

In [ ]:
def build_prompt(query_item, few_shot_items):
    label_str = ", ".join(LABELS)

    instruction = (
        "You are an expert in biomedical relation extraction.\n"
        "Determine the relation between Entity 1 (Head) and Entity 2 (Tail).\n"
        f"Choose exactly one label from this list: [{label_str}]\n"
        "Return only the label.\n\n"
    )

    examples = ""
    for i, ex in enumerate(few_shot_items, 1):
        examples += f"Example {i}\n{format_example(ex, include_label=True)}\n"

    query = f"Now classify the following example.\n\n{format_example(query_item, include_label=False)}"
    return instruction + examples + query

In [ ]:
with open(OUTPUT_PREPARED, "w", encoding="utf-8") as fout:
    for item in tqdm(dev_data, desc="Preparing prompts"):
        few_shot = retrieve_few_shot_examples(item, k=K_SHOTS)

        marked_text = build_marked_text(item)
        few_shot_marked = [build_marked_text(x) for x in few_shot]
        prompt = build_prompt(item, few_shot)

        row = {
            "document_id": get_doc_id(item),
            "gold_label": get_relation(item),

            "input_text": item["text"],
            "input_marked_text": marked_text,

            "head_text": get_head_name(item),
            "head_pos": get_head_pos(item),
            "head_span": get_head_span(item),
            "head_type": get_head_type(item),

            "tail_text": get_tail_name(item),
            "tail_pos": get_tail_pos(item),
            "tail_span": get_tail_span(item),
            "tail_type": get_tail_type(item),

            "pair_type": f"{get_head_type(item)}->{get_tail_type(item)}",

            "few_shot_labels": [get_relation(x) for x in few_shot],
            "few_shot_doc_ids": [get_doc_id(x) for x in few_shot],
            "few_shot_examples_json": few_shot,
            "few_shot_marked_texts": few_shot_marked,

            "prompt": prompt,
            "raw_input_json": item,
        }

        fout.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Prepared prompts saved to: {OUTPUT_PREPARED}")

Preparing prompts: 100%|██████████| 2967/2967 [00:33<00:00, 87.59it/s] 

Prepared prompts saved to: prepared_dev_6.jsonl
